In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

## Load data

In [2]:
df_train = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_train.csv")
df_test = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_test.csv")

In [3]:
columns_remove = [
    'VitaminD',
    'YearStart',
]

In [4]:
df_train = df_train[df_train['milk_consumption']<=3]
df_test = df_test[df_test['milk_consumption']<=3]

In [5]:
df_train.drop(columns=columns_remove, inplace=True)
df_test.drop(columns=columns_remove, inplace=True)

In [6]:
category_columns = [
    'Gender','SmokeFam' ,'Race', 'label','milk_consumption'
]

In [7]:
unuseful_features = ['WaistCircumference','AST','ALT','AlkalinePhosphotase','UricAcid','LDLCholesterol','Hematocrit','MeanCellHemoglobin','PlateletCount', 'MeanPlateletVolume','familysize']

In [8]:
df_train.drop(columns=unuseful_features,inplace=True)
df_test = df_test[df_train.columns]

## Training:  SMOTE with SVMSMOTE

In [9]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# ========== 0) Assume dataset split ==========
# X_train_raw, X_test_raw, y_train, y_test
# categorical_cols, numeric_cols
X_train_raw = df_train.drop(columns='label')
y_train=df_train['label']
X_test_raw=df_test.drop(columns=['label'])
y_test=df_test['label']
categorical_cols = ['Gender','Race', 'milk_consumption','SmokeFam']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col !='label']
# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

# ====== 2) Classifiers ======
#lightGBM: {'classifier__learning_rate': 0.05, 'classifier__max_depth': 6, 'classifier__n_estimators': 120, 'classifier__num_leaves': 31}
classifiers = {
    'LightGBM': LGBMClassifier(n_estimators=120,max_depth= 6,num_leaves=31, learning_rate=0.05, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=80,learning_rate=0.066, random_state=42, verbosity=0),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)
}

# ====== 3) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='auto',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 4) Wrapper class for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

# ====== 5) Train and evaluate models ======
results = []

for name, clf in classifiers.items():
    print(f"\n🚀 Training {name} with SVMSMOTE...")
    
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', svmsmote),
        ('classifier', clf)
    ])
    
    wrapped_model = ImblearnWrapper(pipeline)
    
    try:
        wrapped_model.fit(X_train_raw, y_train)
        y_pred = wrapped_model.predict(X_test_raw)
        y_proba = wrapped_model.predict_proba(X_test_raw)
        
        if len(np.unique(y_test)) == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
        
        precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        accuracy = accuracy_score(y_test, y_pred)
        p_class, r_class, f1_class, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )

        print("Per-class metrics:")
        for i in range(len(p_class)):
            print(f"Class {i}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")
        results.append({
            'Model': name,
            'Precision (Macro)': precision,
            'Recall (Macro)': recall,
            'F1 Score (Macro)': f1,
            'Accuracy': accuracy,
            'AUC': auc
        })
        
        print(f"✅ {name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
        
    except Exception as e:
        print(f"❌ Error training {name}: {e}")

# ====== 6) Save results ======
results_df = pd.DataFrame(results)
print("\n📊 FINAL RESULTS SUMMARY:")
print("="*60)
print(results_df.to_string(index=False, float_format='%.4f'))

results_df.to_csv("svmsmote_classifiers_results.csv", index=False)
print("\n✅ Results exported to: svmsmote_classifiers_results.csv")

# ====== 7) Find best model ======
best_idx = results_df['F1 Score (Macro)'].idxmax()
best_model = results_df.loc[best_idx, 'Model']
best_f1 = results_df.loc[best_idx, 'F1 Score (Macro)']
print(f"\n🏆 BEST MODEL: {best_model} (F1 Score: {best_f1:.4f})")



🚀 Training LightGBM with SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002703 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4848
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Per-class metrics:
Class 0: P=0.8157, R=0.7485, F1=0.7807, Support=3193
Class 1: P=0.5276, R=0.6242, F1=0.5719, Support=1437
✅ XGBoost - Accuracy: 0.7099, F1: 0.6763, AUC: 0.7442

🚀 Training LogisticRegression with SVMSMOTE...
Per-class metrics:
Class 0: P=0.8447, R=0.5759, F1=0.6849, Support=3193
Class 1: P=0.4480, R=0.7648, F1=0.5650, Support=1437
✅ LogisticRegression - Accuracy: 0.6346, F1: 0.6250, AUC: 0.7309

🚀 Training RandomForest with SVMSMOTE...
Per-class metrics:
Class 0: P=0.7797, R=0.8259, F1=0.8021, Support=3193
Class 1: P=0.5545, R=0.4816, F1=0.5155, Support=1437
✅ RandomForest - Accuracy: 0.7190, F1: 0.6588, AUC: 0.7305

🚀 Training GradientBoosting with SVMSMOTE...
Per-class metrics:
Class 0: P=0.8211, R=0.6959, F1=0.7533, Support=3193
Class 1: P=0.4953, R=0.6632, F1=0.5671, Support=1437
✅ GradientBoosting - Accuracy: 0.6857, F1: 0.6602, AUC: 0.7412

🚀 Training KNN with SVMSMOTE...
Per-class metrics:
Class 0: P=0.7753, R=0.5878, F1=0.6687, Support=3193
Class 1: P=0.4043,

In [54]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# ========== 0) Assume dataset split ==========
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption','SmokeFam']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col != 'label']

# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)
dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}

# ====== 2) Base classifiers for stacking ======
# base_learners = [
#     ('lightgbm', LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
#     ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
#     ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42)),
# ]
from sklearn.ensemble import GradientBoostingClassifier

gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}
#Best: GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)
base_learners = [
    ('lightgbm', LGBMClassifier(
        n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    ('xgboost', XGBClassifier(
        n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
    ('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)),
    ('dt', dt_setups['Shallow']),  
    ('nb', GaussianNB()),  
]
# ====== 3) Meta-learner ======
# meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    penalty='elasticnet',   # mix between L1 and L2
    l1_ratio=0.5,           # 0.0 -> pure L2, 1.0 -> pure L1
    C=1.0,                  # regularization strength
    class_weight='balanced',
    random_state=42
)
# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  # use probabilities from base models
    n_jobs=-1
)

# ====== 5) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='auto',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 6) Full pipeline with SVMSMOTE + Stacking ======
stacking_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', svmsmote),
    ('classifier', stacking_clf)
])

# ====== 7) Train and evaluate ======
print("\n🚀 Training Ensemble Stacking Model with SVMSMOTE...")
stacking_pipeline.fit(X_train_raw, y_train)

y_pred = stacking_pipeline.predict(X_test_raw)
y_proba = stacking_pipeline.predict_proba(X_test_raw)

# Evaluate
if len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba[:, 1])
else:
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
accuracy = accuracy_score(y_test, y_pred)
p_class, r_class, f1_class, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)
print(f1_w)
print("\n📊 Per-class metrics:")
for i in range(len(p_class)):
    print(f"Class {i}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")

print("\n✅ Ensemble Stacking Results")
print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")



🚀 Training Ensemble Stacking Model with SVMSMOTE...
0.7220838430860415

📊 Per-class metrics:
Class 0: P=0.8093, R=0.7748, F1=0.7917, Support=3193
Class 1: P=0.5429, R=0.5943, F1=0.5674, Support=1437

✅ Ensemble Stacking Results
Accuracy: 0.7188, F1: 0.6796, AUC: 0.7446


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
